# 04 — Ablations

Sweeps over the knobs in `configs/base.yaml`. Each sweep re-runs the relevant stage
with an override and collects the metric of interest.

Sweeps covered: vocab size, embedding init method, LoRA rank, `modules_to_save`,
token-selection strategy, RQ-VAE (levels L / codebook size K), and SID-vs-freq vocab.

In [ ]:
import subprocess, json, sys, pathlib, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd().parent))
CONFIG = '../configs/experiments/vocab_qlora.yaml'

def run(module, overrides):
    subprocess.run([sys.executable, '-m', module, CONFIG, *overrides], check=True, cwd='..')

In [ ]:
# 9a. Vocabulary size sweep: compression vs. number of new tokens (CPU-only stage)
rows = []
for n in [32, 64, 128, 256, 512, 1024]:
    run('src.eval_compression', [f'mining.top_n={n}'])
    r = json.load(open('../results/compression_vocab_qlora.json'))
    rows.append({'top_n': n, 'reduction_pct': r['reduction_pct'].get('both') or r['reduction_pct'].get('freq')})
pd.DataFrame(rows).plot(x='top_n', y='reduction_pct', marker='o', title='Compression vs vocab size')

## Remaining sweeps (GPU)

Each is a `train_qlora` + `eval_*` pair with one override; collect metrics as above.

- **9b init method**: `extend.init_method` in `[random, mean, exp_weighted, codebook]`
- **9c LoRA rank**: `qlora.lora_r` in `[8, 16, 32, 64]` (with/without vocab ext)
- **9d modules_to_save**: `extend.modules_to_save=[]` vs `[embed_tokens,lm_head]`
- **9e selection strategy**: `mining.strategies` variants + a random-token control
- **9f RQ-VAE**: `semantic_ids.rqvae.levels` in `[2,3,4]`, `...codebook_size` in `[32,64,128,256]`
- **9g SID vs freq**: compare `semid_only` / `vocab_only` / `semid_qlora` token-adoption